# 20 — CNN 前処理多様性アンサンブル

ET と CNN の相関が r=0.9902 と高く、ブレンド効果なし。
異なる前処理で学習した CNN を組み合わせることで多様性を確保する。

| 前処理 | 変換内容 |
|---|---|
| PP_A | SNV → SG1(41,3,1) [nb18 と同じ] |
| PP_B | MSC(ref=fold訓練平均) → SG1(41,3,1) |
| PP_C | CR_div (上凸包包絡線比) |

Seed=42 固定。多様性は前処理から。
nb18 3-seed: CV=21.96%, LB=17.73

In [ ]:
import sys, os, copy
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from src.utils import load_data, parse_spectra, get_groups, make_submission

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
CLIP_T = 200.0

train_df, test_df = load_data()
train_meta, y_s, X_raw, wn = parse_spectra(train_df)
test_meta,  _,   X_test_raw, _ = parse_spectra(test_df)
y      = y_s.values.astype(float)
groups = get_groups(train_meta)
SPLITS = list(GroupKFold(n_splits=5).split(X_raw, y, groups))

print(f'Device: {DEVICE}')
print(f'Train: {X_raw.shape}  Test: {X_test_raw.shape}')

In [ ]:
# ===== ImprovedCNN1D (same as nb17-19) =====
class ImprovedCNN1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.block12 = nn.Sequential(
            nn.Conv1d(1,  8,  kernel_size=15, padding=7), nn.BatchNorm1d(8),  nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(8,  16, kernel_size=9,  padding=4), nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.conv3     = nn.Sequential(
            nn.Conv1d(16, 32, kernel_size=5, padding=2), nn.BatchNorm1d(32))
        self.shortcut3 = nn.Conv1d(16, 32, kernel_size=1)
        self.pool = nn.AdaptiveAvgPool1d(16)
        self.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(32 * 16, 32), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        h = self.block12(x.unsqueeze(1))
        h = torch.relu(self.conv3(h) + self.shortcut3(h))
        h = self.pool(h)
        return self.fc(h.view(x.size(0), -1)).squeeze(1)

print(f'Params: {sum(p.numel() for p in ImprovedCNN1D().parameters()):,}')

# ===== 前処理関数 (すべて fold 内で fit on train のみ) =====

def pp_snv_sg1(Xtr, Xva):
    """SNV (per-spectrum) -> SG1(41,3,1)"""
    def _apply(X):
        A = X.astype(float)
        A = (A - A.mean(1, keepdims=True)) / (A.std(1, keepdims=True) + 1e-8)
        return savgol_filter(A, 41, 3, deriv=1, axis=1).astype(np.float32)
    return _apply(Xtr), _apply(Xva)

def pp_msc_sg1(Xtr, Xva):
    """MSC (ref=Xtr mean, fit on train) -> SG1(41,3,1)"""
    ref = Xtr.mean(axis=0).astype(float)
    ref_c = ref - ref.mean()
    denom = (ref_c ** 2).sum()
    def _apply(X):
        A = X.astype(float)
        Ac = A - A.mean(axis=1, keepdims=True)
        slope = (Ac @ ref_c) / denom          # (n,)
        intercept = A.mean(axis=1) - slope * ref.mean()  # (n,)
        out = (A - intercept[:, None]) / (slope[:, None] + 1e-8)
        return savgol_filter(out, 41, 3, deriv=1, axis=1).astype(np.float32)
    return _apply(Xtr), _apply(Xva)

def _upper_hull_interp(wn_s, R_s):
    hull = []
    for i in range(len(wn_s)):
        while len(hull) >= 2:
            o, a = hull[-2], hull[-1]
            cross = ((wn_s[a]-wn_s[o])*(R_s[i]-R_s[o])
                     - (R_s[a]-R_s[o])*(wn_s[i]-wn_s[o]))
            if cross >= 0:
                hull.pop()
            else:
                break
        hull.append(i)
    return np.interp(wn_s, wn_s[hull], R_s[hull])

def pp_crdiv(Xtr, Xva):
    """CR_div = R / convex_hull(R) (per-spectrum, no reference needed)"""
    wn_arr = wn.astype(float)
    def _apply(X):
        out = np.zeros_like(X, dtype=np.float32)
        for i, R in enumerate(X.astype(float)):
            h = _upper_hull_interp(wn_arr, R)
            out[i] = (R / np.maximum(h, 1e-8)).astype(np.float32)
        return out
    return _apply(Xtr), _apply(Xva)

PP_CONFIGS = {
    'PP_A_snv_sg1': pp_snv_sg1,
    'PP_B_msc_sg1': pp_msc_sg1,
    'PP_C_crdiv':   pp_crdiv,
}

def rmse_le(yt, yp, T=170.0):
    m = yt <= T
    return float(np.sqrt(np.mean((yt[m]-yp[m])**2))) if m.sum()>0 else np.nan
def rmse_all(yt, yp): return float(np.sqrt(np.mean((yt-yp)**2)))

print('Preprocessing configs:', list(PP_CONFIGS.keys()))

In [ ]:
def train_one(Xtr_pp, ytr, Xva_pp, yva, seed=42,
              n_epochs=100, batch=32, lr=1e-3, patience=20):
    torch.manual_seed(seed)
    np.random.seed(seed)
    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr_pp).astype(np.float32)
    Xva_s = sc.transform(Xva_pp).astype(np.float32)
    Xtr_t = torch.from_numpy(Xtr_s).to(DEVICE)
    ytr_t = torch.from_numpy(ytr.astype(np.float32)).to(DEVICE)
    Xva_t = torch.from_numpy(Xva_s).to(DEVICE)
    yva_t = torch.from_numpy(yva.astype(np.float32)).to(DEVICE)
    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch, shuffle=True)
    model = ImprovedCNN1D().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    crit  = nn.HuberLoss(delta=10.0)
    best_val, best_state, best_preds = float('inf'), None, None
    no_improve = 0
    stop_ep = n_epochs
    for epoch in range(n_epochs):
        model.train()
        for xb, yb in loader:
            loss = crit(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            vp = model(Xva_t)
            vl = crit(vp, yva_t).item()
        if vl < best_val:
            best_val = vl
            best_state = copy.deepcopy(model.state_dict())
            best_preds = vp.cpu().numpy()
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            stop_ep = epoch + 1
            break
    model.load_state_dict(best_state)
    return model, sc, best_preds, stop_ep

print('train_one ready.')

## GroupKFold CV — 3 前処理 × seed=42

In [ ]:
pp_names = list(PP_CONFIGS.keys())

# pp_oof[pp_name] = list of (yva, pred_va) per fold
pp_fold_rmse = {p: [] for p in pp_names}
pp_oof_pred  = {p: [] for p in pp_names}
oof_y_all    = []
stop_ep_all  = {p: [] for p in pp_names}

for fi, (tr, va) in enumerate(SPLITS):
    Xtr_raw, Xva_raw = X_raw[tr], X_raw[va]
    ytr, yva         = y[tr],     y[va]
    oof_y_all.append(yva)

    fold_preds_all = []
    for pname, pp_fn in PP_CONFIGS.items():
        Xtr_pp, Xva_pp = pp_fn(Xtr_raw, Xva_raw)
        _, _, pred, stop_ep = train_one(Xtr_pp, ytr, Xva_pp, yva, seed=SEED)
        r = rmse_le(yva, pred)
        pp_fold_rmse[pname].append(round(r, 2))
        pp_oof_pred[pname].append(pred)
        stop_ep_all[pname].append(stop_ep)
        fold_preds_all.append(pred)
        print(f'  Fold{fi+1} {pname}: RMSE_le170={r:.2f}%  stop_ep={stop_ep}')

    # Combined OOF for this fold
    avg_pred = np.mean(fold_preds_all, axis=0)
    r_comb   = rmse_le(yva, avg_pred)
    print(f'  Fold{fi+1} COMBINED: RMSE_le170={r_comb:.2f}%')
    print()

oof_y = np.concatenate(oof_y_all)

# Combined ensemble OOF
oof_p_combined = np.mean(
    [np.concatenate(pp_oof_pred[p]) for p in pp_names], axis=0)

# Per-PP summary
print('=== CV Summary ===')
print(f'{"":<20} F1     F2     F3     F4     F5   Mean')
for p in pp_names:
    folds = pp_fold_rmse[p]
    print(f'{p:<20} {folds[0]:5.2f}  {folds[1]:5.2f}  {folds[2]:5.2f}  {folds[3]:5.2f}  {folds[4]:5.2f}  {np.mean(folds):5.2f}%')

comb_le = rmse_le(oof_y, oof_p_combined)
print(f'{"COMBINED":<20}                                   {comb_le:5.2f}% (overall OOF)')
print()
print('nb18 reference (SNV+SG1 x3 seed):  21.96%')
print()
print('OOF ensemble distribution:')
print(f'  min={oof_p_combined.min():.1f}  mean={oof_p_combined.mean():.1f}  '
      f'max={oof_p_combined.max():.1f}  >170: {(oof_p_combined>170).sum()}')

# Pairwise prediction correlations (OOF)
print()
print('OOF 予測相関 (y<=170 mask):')
mask170 = oof_y <= 170
for i, pi in enumerate(pp_names):
    pi_pred = np.concatenate(pp_oof_pred[pi])
    for j, pj in enumerate(pp_names):
        if j <= i: continue
        pj_pred = np.concatenate(pp_oof_pred[pj])
        r = np.corrcoef(pi_pred[mask170], pj_pred[mask170])[0,1]
        print(f'  {pi} vs {pj}: r={r:.4f}')

In [ ]:
import os
os.makedirs('../results', exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Per-fold RMSE
ax = axes[0]
x = np.arange(5)
colors = ['steelblue', 'darkorange', 'seagreen']
for i, (p, col) in enumerate(zip(pp_names, colors)):
    offset = (i - 1) * 0.25
    ax.bar(x + offset, pp_fold_rmse[p], 0.25, label=p.split('_',1)[1], color=col, alpha=0.8)
ax.axhline(21.96, color='gray', ls='--', lw=0.8, label='nb18 (21.96%)')
ax.set_xticks(x); ax.set_xticklabels([f'F{i+1}' for i in range(5)])
ax.set_ylabel('RMSE_le170 (%)'); ax.set_title('Per-fold by preprocessing')
ax.legend(fontsize=7); ax.grid(True, alpha=0.3, axis='y')

# OOF scatter (combined)
ax = axes[1]
m = oof_y <= 170
ax.scatter(oof_y[m], oof_p_combined[m], s=4, alpha=0.4)
lim = max(oof_y[m].max(), oof_p_combined[m].max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', lw=0.8)
ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
ax.set_title(f'OOF combined (overall RMSE_le170={comb_le:.2f}%)')
ax.grid(True, alpha=0.3)

# Prediction distributions by PP
ax = axes[2]
for p, col in zip(pp_names, colors):
    pp_pred = np.concatenate(pp_oof_pred[p])
    ax.hist(pp_pred[m], bins=40, alpha=0.4, color=col, label=p.split('_',1)[1])
ax.set_xlabel('Prediction (%)'); ax.set_title('OOF pred distribution (y<=170)')
ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/nb20_pp_ensemble_cv.png', dpi=110)
plt.close()
print('Saved: results/nb20_pp_ensemble_cv.png')

## Full Train → Test Prediction

In [ ]:
print('=== Test predictions: 3 PP on full train ===')

te_preds = {}
for pname, pp_fn in PP_CONFIGS.items():
    avg_stop = int(np.mean(stop_ep_all[pname]))
    print(f'{pname}: avg_stop={avg_stop}')

    Xtr_pp, Xte_pp = pp_fn(X_raw, X_test_raw)

    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr_pp).astype(np.float32)
    Xte_s = sc.transform(Xte_pp).astype(np.float32)
    Xtr_t = torch.from_numpy(Xtr_s).to(DEVICE)
    ytr_t = torch.from_numpy(y.astype(np.float32)).to(DEVICE)

    torch.manual_seed(SEED)
    np.random.seed(SEED)
    model_f = ImprovedCNN1D().to(DEVICE)
    opt_f   = torch.optim.Adam(model_f.parameters(), lr=1e-3)
    sched_f = torch.optim.lr_scheduler.CosineAnnealingLR(opt_f, T_max=100)
    crit_f  = nn.HuberLoss(delta=10.0)
    loader_f = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=32, shuffle=True)
    for ep in range(avg_stop):
        model_f.train()
        for xb, yb in loader_f:
            loss = crit_f(model_f(xb), yb)
            opt_f.zero_grad(); loss.backward(); opt_f.step()
        sched_f.step()
    model_f.eval()
    with torch.no_grad():
        Xte_t  = torch.from_numpy(Xte_s).to(DEVICE)
        te_p   = model_f(Xte_t).cpu().numpy()
    te_preds[pname] = te_p
    tr_rmse = torch.sqrt(torch.mean((model_f(Xtr_t)-ytr_t)**2)).item()
    print(f'  train_rmse={tr_rmse:.2f}  test: min={te_p.min():.1f}  mean={te_p.mean():.1f}  max={te_p.max():.1f}  >170: {(te_p>170).sum()}')

# Equal-weight ensemble
te_ensemble = np.clip(
    np.mean(list(te_preds.values()), axis=0), 0, CLIP_T)
print()
print(f'ENSEMBLE: min={te_ensemble.min():.1f}  mean={te_ensemble.mean():.1f}  '
      f'max={te_ensemble.max():.1f}  >170: {(te_ensemble>170).sum()}')

# Also check pairwise correlation
pnames = list(te_preds.keys())
print()
print('テスト予測 pairwise 相関:')
for i in range(len(pnames)):
    for j in range(i+1, len(pnames)):
        r = np.corrcoef(te_preds[pnames[i]], te_preds[pnames[j]])[0,1]
        print(f'  {pnames[i]} vs {pnames[j]}: r={r:.4f}')
print(f'  vs ET (s13): ...(check at submission time)')

In [ ]:
import os
os.makedirs('../submissions', exist_ok=True)

make_submission(test_meta, te_ensemble, '../submissions/sub_cnn_pp_ensemble.csv')
print('Saved: submissions/sub_cnn_pp_ensemble.csv')

print()
print('=== Final Summary ===')
print(f'{"前処理":<22} {"Folds":40} Mean')
for p in pp_names:
    folds = pp_fold_rmse[p]
    print(f'{p:<22} {str(folds):40} {np.mean(folds):.2f}%')
print(f'{"COMBINED (overall OOF)":<22} {"":40} {comb_le:.2f}%')
print()
print('Reference:')
print('  nb18 SNV+SG1 x3seed: CV=21.96%  LB=17.73')
print()
print('Test distribution:')
print(f'  ENSEMBLE: mean={te_ensemble.mean():.1f}%  >170: {(te_ensemble>170).sum()}')